# 01b. Precomputed extracellular potentials

In the previous tutorial, we introduced extracellular stimulation and recording in Dendra and the core theoretical concepts. In that tutorial, we computed the field using an analytic model (a point source in an infinite homogenous medium). However, in many cases, the field may be more complex and not amenable to an analytic solution. For example, the field may be generated by a complex electrode geometry, or the medium may be inhomogeneous and anisotropic. In these cases, we can use precomputed fields that are generated from numerical simulations (e.g., finite element method) or from experimental measurements. All the same principles of quasistaticity, supoerposition, and reciprocity still apply, but we need to introduce functionality to consume precomputed field data. In this tutorial, we will show how to use precomputed fields in Dendra.


## Core classes

The core classes for working with precomputed fields are:
- {py:class}`~dendra.models.fields.precomputed_interpolate_1d`: This class implements 1D interpolation of precomputed fields. This is useful for fields that are sampled along a single spatial dimension (e.g., along the length of an axon).
- {py:class}`~dendra.models.fields.precomputed_interpolate_3d_rect`: This class implements 3D interpolation of precomputed fields on a rectilinear grid. This is useful for fields for which you have dense sampling in three dimensions on a consistent grid.
- {py:class}`~dendra.models.fields.precomputed_interpolate_3d_scattered`: This class implements interpolation of precomputed fields at scattered points in 3D space. This is useful for fields for which you have sampling at arbitrary locations in 3D space (e.g., from experimental measurements, or from simulations with irregular / non-rectilinear meshes).
- {py:class}`~dendra.models.fields.efield_interpolate_3d_rect`: This class implements 3D interpolation of precomputed electric fields on a rectilinear grid, and computes the quasipotentials at the model's coordinates. This is useful for computing extracellular stimulation using precomputed E-field data (e.g., in TMS where you solve for the E-field, not potentials).
- {py:class}`~dendra.models.fields.efield_interpolate_3d_scattered`: This class implements interpolation of precomputed electric fields at scattered points in 3D space, and computes the quasipotentials at the model's coordinates. This is useful for computing extracellular stimulation using precomputed E-field data that is sampled at arbitrary locations in 3D space (e.g., from simulations with irregular / non-rectilinear meshes).

:::{important}
If you use the [ASCENT pipeline](https://wmglab-duke-ascent.readthedocs.io/en/latest/), you can consume fields generated from the pipeline's FEM simulations using `precomputed_interpolate_1d.from_ascent`:

```python
from dendra.models.fields import precomputed_interpolate_1d
field_model = precomputed_interpolate_1d.from_ascent(
    ascent_dir="path/to/ascent/",
    sample=0, model=0, sim=0, contact=0
)
```
:::

In all cases, their use follows the same pattern (as with the analytic field classes):
1. Prepare the field model by providing the precomputed data and sample points.
2. Call the field model on the neuron model to compute the field values at the model's coordinates.
3. Use the computed field values for extracellular stimulation or recording as needed.


## Example: Extracellular stimulation of axon with precomputed fields

We will first simulate a single myelinated axon model receiving extracellular kHz stimulation. For this example, we will assume that we have the field stored as a function of arc length along the axon (e.g., from a 1D cable model of the field along the axon). We will use the {py:class}`~dendra.models.fields.precomputed_interpolate_1d` class to consume this data and compute the potentials at the model's coordinates.

In [ ]:
import numpy as np
import dendra as dn
from dendra.models.mod import hh
from dendra.units import mm, um

model = dn.Myelinated(
    diameters=[6.0], n_node=201, node_length=1.0, celsius=6.3, v_init=-65.0
)
model.insert(hh)

In [ ]:
data = np.load('0.npy') * 1000 # in mV
xs = np.load('fiber_xs.npy') # in um

field_model = dn.precomputed_interpolate_1d(
    data = data, # 2D array of shape (n_fiber_locations, n_points_along_axon), in mV
    x = xs, # 1D array of shape (n_points_along_axon,)
)

Dendra's `precomputed_interpolate_1d` class assumes that the field is sampled along the arc length of the axon, and that the field values are stored in a 2D array of shape (n_fiber_locations, n_points_along_axon), where n_fiber_locations is the number of fiber locations for which you have precomputed data, and n_points_along_axon is the number of points along the axon where you have sampled the field. The `x` array should contain the corresponding x-coordinates (arc lengths) for the sampled points along the axon - this may be 1D (in which case the same x-coordinates are used for all fiber locations) or 2D (allowing different x-coordinates for each fiber location).

In [ ]:
print(data.shape, xs.shape)

You see that we have data for 46 fiber locations, and the field is sampled at 7499 points along the axon for each location. If we call the field model as is on the axon model, it will raise an error because the field model does not know which fiber location to use for our axon model (since we have 46 different fiber locations in our data, but only one axon model). To resolve this, we can specify the fiber location index to use when we call the field model. For example, if we want to use the field data from the first fiber location, we can do:

In [ ]:
ve = field_model(model, 0)

(If we had a batch of models, we could specify a different fiber location index for each by using a list / array of indices).

Let's visualize the field along the axon for this fiber location:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(model.x[0], ve[0])
plt.xlabel("Arc length along axon")
plt.ylabel("Field value (mV)")
plt.title("Field along axon for first fiber location")
plt.show()

As you can see, the field peak is offset from the centre of the axon: by default, axon models in Dendra are centered at the origin, but the field data may be generated for axons that are not centered at the origin. In this case, we can see that xs starts from 10 um - we can align the fields by simply offsetting the x-coordinates of the field data:

In [ ]:
xs = xs - xs[xs.shape[0] // 2] # offset x-coordinates to align field peak with center of axon
field_model = dn.precomputed_interpolate_1d(
    data = data, # 2D array of shape (n_fiber_locations, n_points_along_axon), in mV
    x = xs, # 1D array of shape (n_points_along_axon,)
)
ve = field_model(model, 0)

plt.plot(model.x[0], ve[0])
plt.xlabel("Arc length along axon")
plt.ylabel("Field value (mV)")
plt.title("Field along axon for first fiber location (aligned)")
plt.show()

In [ ]:
from dendra.units import mA, kHz

i_stim = dn.sin(amp=0.2 * mA, freq=1.0 * kHz)  # 0.1 mA, 1 kHz sine wave
rec = dn.callbacks.Recorder(['v'], node_indices=model.c(0.25, 0.5, 0.75, 1.0))

tstop, dt = 50.0, 0.001

# run as before, but supply extracellular potential for stimulation
rec.reset()
model.initialize()
model.longrun(
    tstop=tstop, dt=dt, chunklength=100, 
    extra=(ve, i_stim), callbacks=[rec],
    progressbar=False
)

In [ ]:
v = rec.numpy('v')
t = np.arange(0, tstop+dt, dt)

plt.figure(figsize=(5, 3), dpi=200)
plt.plot(t, v[:, 0, 0], label="25% node")
plt.plot(t, v[:, 0, 1], label="under stimulus")
plt.plot(t, v[:, 0, 2], label="75% node")
plt.plot(t, v[:, 0, 3], label="end node")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane Potential (mV)")
plt.legend()
plt.show()

That's it! You now know how to use precomputed field data in Dendra. You can use the same principles to work with 3D field data using the other precomputed field classes.